In [1]:
import findspark
findspark.init("/opt/manual/spark/")

In [2]:
from pyspark.sql import SparkSession, functions as F

In [3]:
spark = SparkSession.builder \
.appName("Simple ML") \
.master("local[2]") \
.getOrCreate()

2022-07-22 12:37:28,766 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
! wget -O ~/datasets/Mall_Customers.csv  https://raw.githubusercontent.com/erkansirin78/datasets/master/Mall_Customers.csv

--2022-07-22 12:37:38--  https://raw.githubusercontent.com/erkansirin78/datasets/master/Mall_Customers.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4365 (4.3K) [text/plain]
Saving to: ‘/home/train/datasets/Mall_Customers.csv’

100%[======================================>] 4,365       --.-K/s   in 0.001s  

2022-07-22 12:37:39 (4.08 MB/s) - ‘/home/train/datasets/Mall_Customers.csv’ saved [4365/4365]



In [5]:
df = spark.read.format("csv") \
.option("header",True) \
.option("inferSchema",True) \
.option("sep",",") \
.load("file:///home/train/datasets/Mall_Customers.csv")

In [6]:
df.show(5)

+----------+------+---+------------+-------------+
|CustomerID|Gender|Age|AnnualIncome|SpendingScore|
+----------+------+---+------------+-------------+
|         1|  Male| 19|       15000|           39|
|         2|  Male| 21|       15000|           81|
|         3|Female| 20|       16000|            6|
|         4|Female| 23|       16000|           77|
|         5|Female| 31|       17000|           40|
+----------+------+---+------------+-------------+
only showing top 5 rows



In [7]:
df.count()

200

In [10]:
df.groupBy("Gender").count().show()

+------+-----+
|Gender|count|
+------+-----+
|Female|  112|
|  Male|   88|
+------+-----+



# StringIndexer

In [8]:
from pyspark.ml.feature import StringIndexer

In [9]:
gender_stridx_obj = StringIndexer(inputCol="Gender", 
                                  outputCol="genderIdx", 
                                  handleInvalid='error')

In [10]:
df1 = gender_stridx_obj.fit(df).transform(df)

In [11]:
df1.show(5)

+----------+------+---+------------+-------------+---------+
|CustomerID|Gender|Age|AnnualIncome|SpendingScore|genderIdx|
+----------+------+---+------------+-------------+---------+
|         1|  Male| 19|       15000|           39|      1.0|
|         2|  Male| 21|       15000|           81|      1.0|
|         3|Female| 20|       16000|            6|      0.0|
|         4|Female| 23|       16000|           77|      0.0|
|         5|Female| 31|       17000|           40|      0.0|
+----------+------+---+------------+-------------+---------+
only showing top 5 rows



# OneHotEncoder

In [13]:
from pyspark.ml.feature import OneHotEncoder

In [14]:
ohe_obj = OneHotEncoder(inputCols=["genderIdx"],
    outputCols=["OHE_Cols"],
    handleInvalid='error')

In [15]:
df2 = ohe_obj.fit(df1).transform(df1)

In [16]:
df2.show(5)

+----------+------+---+------------+-------------+---------+-------------+
|CustomerID|Gender|Age|AnnualIncome|SpendingScore|genderIdx|     OHE_Cols|
+----------+------+---+------------+-------------+---------+-------------+
|         1|  Male| 19|       15000|           39|      1.0|    (1,[],[])|
|         2|  Male| 21|       15000|           81|      1.0|    (1,[],[])|
|         3|Female| 20|       16000|            6|      0.0|(1,[0],[1.0])|
|         4|Female| 23|       16000|           77|      0.0|(1,[0],[1.0])|
|         5|Female| 31|       17000|           40|      0.0|(1,[0],[1.0])|
+----------+------+---+------------+-------------+---------+-------------+
only showing top 5 rows



In [22]:
df2.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- AnnualIncome: integer (nullable = true)
 |-- SpendingScore: integer (nullable = true)
 |-- genderIdx: double (nullable = false)
 |-- OHE_Cols: vector (nullable = true)



In [17]:
from pyspark.ml.feature import VectorAssembler

In [18]:
assembler_obj = VectorAssembler(inputCols=['Age','AnnualIncome','OHE_Cols'], 
                                outputCol='vector_assebmled_cols', 
                                handleInvalid='skip')

In [19]:
df3 = assembler_obj.transform(df2)

In [20]:
df3.limit(5).toPandas()

,CustomerID,Gender,Age,AnnualIncome,SpendingScore,genderIdx,OHE_Cols,vector_assebmled_cols
0,1,Male,19,15000,39,1.0,(0.0),"[19.0, 15000.0, 0.0]"
1,2,Male,21,15000,81,1.0,(0.0),"[21.0, 15000.0, 0.0]"
2,3,Female,20,16000,6,0.0,(1.0),"[20.0, 16000.0, 1.0]"
3,4,Female,23,16000,77,0.0,(1.0),"[23.0, 16000.0, 1.0]"
4,5,Female,31,17000,40,0.0,(1.0),"[31.0, 17000.0, 1.0]"


In [27]:
df3.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- AnnualIncome: integer (nullable = true)
 |-- SpendingScore: integer (nullable = true)
 |-- genderIdx: double (nullable = false)
 |-- OHE_Cols: vector (nullable = true)
 |-- vector_assebmled_cols: vector (nullable = true)



In [21]:
from pyspark.ml.feature import StandardScaler

In [22]:
scaler = StandardScaler(withMean=False, 
                        withStd=True, 
                        inputCol='vector_assebmled_cols', 
                        outputCol='features')

In [23]:
df4 = scaler.fit(df3).transform(df3)

In [32]:
df4.limit(5).toPandas()

,CustomerID,Gender,Age,AnnualIncome,SpendingScore,genderIdx,OHE_Cols,vector_assebmled_cols,features
0,1,Male,19,15000,39,1.0,(0.0),"[19.0, 15000.0, 0.0]","[1.360153914235199, 0.5711082903036442, 0.0]"
1,2,Male,21,15000,81,1.0,(0.0),"[21.0, 15000.0, 0.0]","[1.503328010470483, 0.5711082903036442, 0.0]"
2,3,Female,20,16000,6,0.0,(1.0),"[20.0, 16000.0, 1.0]","[1.431740962352841, 0.6091821763238872, 2.0095..."
3,4,Female,23,16000,77,0.0,(1.0),"[23.0, 16000.0, 1.0]","[1.6465021067057672, 0.6091821763238872, 2.009..."
4,5,Female,31,17000,40,0.0,(1.0),"[31.0, 17000.0, 1.0]","[2.2191984916469036, 0.64725606234413, 2.00951..."


In [27]:
df5 = df4.withColumnRenamed("SpendingScore", "label")

In [28]:
df5.limit(5).toPandas()

,CustomerID,Gender,Age,AnnualIncome,label,genderIdx,OHE_Cols,vector_assebmled_cols,features
0,1,Male,19,15000,39,1.0,(0.0),"[19.0, 15000.0, 0.0]","[1.360153914235199, 0.5711082903036442, 0.0]"
1,2,Male,21,15000,81,1.0,(0.0),"[21.0, 15000.0, 0.0]","[1.503328010470483, 0.5711082903036442, 0.0]"
2,3,Female,20,16000,6,0.0,(1.0),"[20.0, 16000.0, 1.0]","[1.431740962352841, 0.6091821763238872, 2.0095..."
3,4,Female,23,16000,77,0.0,(1.0),"[23.0, 16000.0, 1.0]","[1.6465021067057672, 0.6091821763238872, 2.009..."
4,5,Female,31,17000,40,0.0,(1.0),"[31.0, 17000.0, 1.0]","[2.2191984916469036, 0.64725606234413, 2.00951..."


# Split data

In [29]:
df_train, df_test = df5.randomSplit([0.8, 0.2], seed=142)

In [30]:
print(df_train.count())
print(df_test.count())

161
39


# Model

In [31]:
from pyspark.ml.regression import DecisionTreeRegressor

In [32]:
estimator = DecisionTreeRegressor(maxBins=28)

In [33]:
gbt_model = estimator.fit(df_train)

In [35]:
transformed_df = gbt_model.transform(df_test)

In [36]:
transformed_df.limit(8).toPandas()

,CustomerID,Gender,Age,AnnualIncome,label,genderIdx,OHE_Cols,vector_assebmled_cols,features,prediction
0,4,Female,23,16000,77,0.0,(1.0),"[23.0, 16000.0, 1.0]","[1.6465021067057672, 0.6091821763238872, 2.009...",76.866667
1,9,Male,64,19000,3,1.0,(0.0),"[64.0, 19000.0, 0.0]","[4.581571079529091, 0.723403834384616, 0.0]",15.000000
2,11,Male,67,19000,14,1.0,(0.0),"[67.0, 19000.0, 0.0]","[4.796332223882017, 0.723403834384616, 0.0]",15.000000
3,15,Male,37,20000,13,1.0,(0.0),"[37.0, 20000.0, 0.0]","[2.6487207803527557, 0.761477720404859, 0.0]",53.388889
4,25,Female,54,28000,14,0.0,(1.0),"[54.0, 28000.0, 1.0]","[3.865700598352671, 1.0660688085668024, 2.0095...",4.000000
5,29,Female,40,29000,31,0.0,(1.0),"[40.0, 29000.0, 1.0]","[2.863481924705682, 1.1041426945870454, 2.0095...",19.200000
6,34,Male,18,33000,92,1.0,(0.0),"[18.0, 33000.0, 0.0]","[1.2885668661175569, 1.2564382386680173, 0.0]",64.750000
7,41,Female,65,38000,35,0.0,(1.0),"[65.0, 38000.0, 1.0]","[4.6531581276467335, 1.446807668769232, 2.0095...",4.000000


# Evaluate the model

In [37]:
from pyspark.ml.evaluation import RegressionEvaluator

In [38]:
evaluator = RegressionEvaluator(metricName='r2')

In [39]:
r2 = evaluator.evaluate(transformed_df)

In [40]:
print(r2)

0.3016155140306942


In [41]:
spark.stop()